[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_91_Phase11_Golden_Datasets_and_LLM_as_Judge.ipynb)

# Lesson 91 — Phase 11 kickoff: Golden Datasets & LLM-as-Judge

**Phase 11 — Evaluation & Trust at Scale · Lesson 1 of ~6**

In Lesson 90 you shipped a real `rag-service` with an 8-case CI gate that checked two things: *did the router pick the right hop count* and *did the answer cite the right doc*. That gate is a **smoke test** — it tells you the service isn't on fire. It does **not** tell you whether your answers are *faithful*, *relevant*, or whether *retrieval actually found the right evidence*.

Phase 11 is about turning "it runs" into "I can prove it's good, and prove it stays good." Today we build the foundation everything else rests on:

1. **Golden datasets** — a curated, versioned set of `(question, ground-truth answer, ground-truth context)` examples. This is your ruler. Without a ruler, every "the new prompt feels better" is a vibe, not a measurement.
2. **The four RAG metrics**, built *from scratch* (RAGAS-style) so you understand what every number means before you `pip install` a library that hides it:
   - **Faithfulness** — is the answer grounded in the retrieved context, or hallucinated?
   - **Answer Relevancy** — does the answer actually address the question?
   - **Context Precision** — of what we retrieved, is the *relevant* stuff ranked high?
   - **Context Recall** — did retrieval find *everything* the ground-truth answer needs?
3. **LLM-as-judge** — using a model to score free-text answers, its systematic **biases**, and how you **calibrate** a judge against human labels (callback to L29).

> **Everything here runs keyless.** We use a deterministic mock for the "LLM judge" and a from-scratch bag-of-words embedder, so the notebook is fully reproducible in Colab with no API key. Every technique has a real-LLM hook you can flip on.

### Roadmap

| # | Topic | Status |
|---|-------|--------|
| L83–L90 | Phase 10 — RAG in production (baseline → capstone `rag-service`) | ✓ done |
| **L91** | **Golden datasets + LLM-as-judge (the 4 metrics from scratch)** | **← you are here** |
| L92 | Regression / canary evals in CI + scoring drift | next |
| L93 | Human-in-the-loop feedback → the data flywheel | |
| L94 | Red-teaming & safety evals (jailbreaks, injection via retrieved docs) | |
| L95 | Cost / latency budgets + load-testing the `/ask` path | |
| L96 | Phase-11 capstone: reusable `agent-evals` harness | |

## 1. Why you cannot ship without a golden dataset

Imagine you tweak your retriever's chunk size (L84) and your answers "seem better." How do you *know*? You have three bad options and one good one:

- **Ship it and watch for complaints.** Users are a slow, noisy, expensive oracle — and by the time they complain, trust is gone.
- **Eyeball 5 outputs.** You will unconsciously cherry-pick, and 5 is far too few to catch a 10% regression.
- **Ask the LLM "is this good?"** with no reference. The judge has nothing to compare against and will rubber-stamp confident nonsense.
- **✅ Score against a golden dataset.** A fixed set of questions where you *already know* the right answer and the right evidence. Now "better" is a number, changes are diffs, and regressions are caught *before* users see them.

A golden row is a contract:

```
{
  "question":       "why is my espresso sour?",         # the input
  "ground_truth":   "Sour espresso means under-extraction: ...",  # the ideal answer
  "ground_context": ["d_sour"],   # which corpus doc(s) SHOULD be retrieved
}
```

**Curation principles** (these matter more than any metric):
- **Cover the distribution, not just the happy path.** Include easy questions, multi-hop questions (L89), out-of-domain questions that *should* be refused (L87 abstention), and known-hard vocabulary-mismatch questions (L88).
- **Ground truth is written by a human who knows the answer**, not generated by the same model you're testing (that just measures self-consistency).
- **Version it.** A golden set is code. It lives in git, it has a changelog, and when you add a row you note *why*.
- **Keep it small and sharp at first.** 20 well-chosen rows beat 2,000 sloppy ones. You will grow it from real failures (that's L93).

## 2. Setup (keyless)

One `pip` cell. We only need `numpy` for a couple of vector ops and `scipy` for a correlation coefficient — both preinstalled in Colab, so this is basically instant. **No API key required.**

In [ ]:
!pip install numpy scipy -q

import os, re, math, json
from dataclasses import dataclass, field
from typing import List, Dict, Callable

# --- Optional real-LLM hook (used ONLY if a key is present; otherwise we mock) ---
def _real_llm_available() -> bool:
    return bool(os.environ.get("OPENAI_API_KEY") or os.environ.get("ANTHROPIC_API_KEY"))

print("Real LLM key detected:", _real_llm_available(), "-> running in",
      "REAL mode" if _real_llm_available() else "MOCK mode (deterministic, keyless)")

## 3. The two primitives every metric needs

Under the hood, our from-scratch metrics lean on exactly two operations:

1. **Tokenize** text into *content words* (lowercase, drop punctuation and stopwords). Stopwords are noise for overlap scoring.
2. **Compare** two spans of text. We use two comparisons:
   - **Overlap coefficient** `|A ∩ B| / min(|A|, |B|)` — *"is A's meaning contained in B?"* Great for "is this claim supported by this chunk?".
   - **Cosine over bag-of-words counts** — *"how topically similar are these two texts?"* Great for answer-relevancy.

We build a tiny bag-of-words embedder. In production you'd swap this for the dense embedder from L85 — the *metric logic is identical*, only the similarity function changes. Keeping it BoW here makes every score hand-checkable.

In [ ]:
STOP = set("a an the is are was were be been being of to in on at for and or "
           "but if then this that these those it its as with your you my i we "
           "why how what when where which do does did can could should would "
           "so too very just about into from by".split())

def tokens(text: str) -> List[str]:
    return [t for t in re.findall(r"[a-z0-9]+", text.lower()) if t not in STOP and len(t) > 1]

def token_set(text: str):
    return set(tokens(text))

def overlap_coeff(a: str, b: str) -> float:
    # overlap coefficient = shared / min-size: is a's content contained in b?
    A, B = token_set(a), token_set(b)
    if not A or not B:
        return 0.0
    return len(A & B) / min(len(A), len(B))

def cosine_bow(a: str, b: str) -> float:
    # cosine similarity over bag-of-words counts; stand-in for a dense embedder (L85)
    from collections import Counter
    ca, cb = Counter(tokens(a)), Counter(tokens(b))
    if not ca or not cb:
        return 0.0
    common = set(ca) & set(cb)
    dot = sum(ca[t] * cb[t] for t in common)
    na = math.sqrt(sum(v * v for v in ca.values()))
    nb = math.sqrt(sum(v * v for v in cb.values()))
    return dot / (na * nb) if na and nb else 0.0

# sanity
print("overlap (contained):", round(overlap_coeff("sour means under extraction",
                                                   "sour espresso is caused by under extraction of the shot"), 3))
print("cosine (topical):   ", round(cosine_bow("grind finer to fix sour shots",
                                               "a finer grind fixes sour under-extracted shots"), 3))

## 4. The corpus and the golden dataset

We reuse the Phase-10 espresso-troubleshooting theme. First a tiny **corpus** (the docs your retriever searches), then the **golden set**: hand-written questions with the ideal answer and which doc(s) *should* be retrieved.

Notice the golden set is deliberately a *mix*:
- rows a good system nails,
- one **out-of-domain** row that *should be refused* (L87),
- and we'll later feed the metrics some *bad* candidate answers/retrievals to prove the metrics actually catch failures.

In [ ]:
CORPUS = {
    "d_sour":   "Sour or sharp espresso is caused by under-extraction. The water passed through too fast. Fix it by grinding finer, using a higher dose, or raising brew temperature.",
    "d_bitter": "Bitter, harsh espresso is caused by over-extraction. The shot ran too slow and pulled too much. Fix it by grinding coarser, lowering the dose, or lowering brew temperature.",
    "d_channel":"Channeling is when water finds a crack and gushes through one spot, giving a fast uneven shot. Prevent it by distributing grounds evenly and tamping level.",
    "d_crema":  "Thin or pale crema usually means stale beans or too coarse a grind. Fresh beans within four weeks of roast and a finer grind restore a thick golden crema.",
    "d_temp":   "Espresso brew temperature should sit between 90 and 96 Celsius. Let the machine warm up for at least twenty minutes so the group head reaches target temperature.",
}

# Each golden row: question, ideal answer, and the doc id(s) that SHOULD be retrieved.
GOLDEN = [
    {"id": "g1", "question": "why is my espresso sour?",
     "ground_truth": "Sour espresso means under-extraction. Fix it by grinding finer, dosing higher, or raising the brew temperature.",
     "ground_context": ["d_sour"]},
    {"id": "g2", "question": "how do I fix a bitter harsh shot?",
     "ground_truth": "Bitter espresso means over-extraction. Grind coarser, lower the dose, or lower the brew temperature.",
     "ground_context": ["d_bitter"]},
    {"id": "g3", "question": "my crema is thin and pale, what is wrong?",
     "ground_truth": "Thin pale crema means stale beans or too coarse a grind. Use fresh beans and grind finer.",
     "ground_context": ["d_crema"]},
    {"id": "g4", "question": "what brew temperature should I use and why does warm-up matter?",
     "ground_truth": "Brew between 90 and 96 Celsius, and warm the machine at least twenty minutes so the group head reaches target temperature.",
     "ground_context": ["d_temp"]},
    {"id": "g5", "question": "is the moon made of cheese?",   # out-of-domain -> should be refused
     "ground_truth": "I don't know based on the provided context.",
     "ground_context": []},
]
print(f"Corpus: {len(CORPUS)} docs | Golden set: {len(GOLDEN)} rows "
      f"({sum(1 for r in GOLDEN if not r['ground_context'])} out-of-domain refusal row)")

## 5. The system under test

To evaluate, we need *candidate outputs* to score: what the system **actually retrieved** and what it **actually answered**. We wire up a transparent retriever (BoW cosine, à la L83/L85) and a deterministic answerer.

Crucially, we also inject **known-bad behaviours** so we can prove the metrics fire:
- `g5` (moon/cheese) — the system wrongly retrieves an espresso doc and confidently hallucinates instead of refusing.
- We keep a separate `SABOTAGE` dict to override outputs for specific rows, simulating regressions.

This mirrors real life: your golden set is fixed; the *system outputs* are what change between versions.

In [ ]:
def retrieve(question: str, k: int = 2) -> List[str]:
    scored = sorted(CORPUS.items(), key=lambda kv: cosine_bow(question, kv[1]), reverse=True)
    return [doc_id for doc_id, _ in scored[:k]]

def answer(question: str, contexts: List[str]) -> str:
    # deterministic 'generator': stitch the top context into a short answer.
    # (Real hook: send question+contexts to an LLM. Here we template for reproducibility.)
    if not contexts:
        return "I don't know based on the provided context."
    return CORPUS[contexts[0]].split(". ")[0] + ". " + CORPUS[contexts[0]].split(". ")[-1]

# Sabotage table: force specific rows to misbehave so metrics have failures to catch.
SABOTAGE = {
    # g5 should refuse; instead the system hallucinates a confident espresso answer.
    "g5": {"contexts": ["d_temp"],
           "answer": "Yes. The moon brews best between 90 and 96 Celsius after a twenty minute warm-up."},
    # g3: retrieval is fine but the generator hallucinates an unsupported claim.
    "g3": {"contexts": ["d_crema"],
           "answer": "Thin crema means your water is too hard, so install a reverse-osmosis filter immediately."},
}

def run_system(row):
    if row["id"] in SABOTAGE:
        s = SABOTAGE[row["id"]]
        ctx = s.get("contexts", retrieve(row["question"]))
        ans = s.get("answer", answer(row["question"], ctx))
    else:
        ctx = retrieve(row["question"])
        ans = answer(row["question"], ctx)
    return {"retrieved_context": ctx, "answer": ans}

for r in GOLDEN:
    out = run_system(r)
    print(f"[{r['id']}] retrieved={out['retrieved_context']}  answer={out['answer'][:70]}...")

## 6. Metric 1 — Faithfulness (is the answer grounded?)

**Question it answers:** *Every claim in the answer — is it actually supported by the retrieved context?* This is your #1 hallucination detector.

**Algorithm (RAGAS-style):**
1. **Decompose** the answer into atomic claims (here: sentences; a real LLM would split "grind finer or dose higher" into two claims).
2. For each claim, check whether it is **entailed** by *any* retrieved chunk. A real judge does NLI-style entailment; our keyless proxy asks *"are this claim's content words contained in some chunk?"* via the overlap coefficient.
3. **Faithfulness = supported claims / total claims** ∈ [0, 1].

A confidently hallucinated answer (like our sabotaged `g3`, "install a reverse-osmosis filter") has claims that appear in **no** chunk → its faithfulness collapses. That's exactly what we want.

In [ ]:
def split_claims(text: str) -> List[str]:
    # Atomic claims ~ sentences. (LLM hook would decompose compound sentences further.)
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p for p in parts if token_set(p)]

def claim_supported(claim: str, contexts: List[str], thr: float = 0.55) -> bool:
    return any(overlap_coeff(claim, CORPUS[c]) >= thr for c in contexts)

def faithfulness(answer_text: str, contexts: List[str]) -> float:
    # A refusal ("I don't know...") is vacuously faithful: it asserts nothing to hallucinate.
    if not token_set(answer_text) or "don't know" in answer_text.lower():
        return 1.0
    claims = split_claims(answer_text)
    if not claims:
        return 1.0
    supported = sum(claim_supported(c, contexts) for c in claims)
    return supported / len(claims)

# grounded answer vs hallucinated answer, same context
good = run_system(GOLDEN[0])            # g1, grounded
bad  = run_system(GOLDEN[2])            # g3, sabotaged hallucination
print("g1 faithfulness (grounded)   :", round(faithfulness(good["answer"], good["retrieved_context"]), 2))
print("g3 faithfulness (hallucinated):", round(faithfulness(bad["answer"],  bad["retrieved_context"]), 2))

# 💡 EXPERIMENT: lower `thr` to 0.3. Faithfulness becomes lenient and the hallucination
# sneaks through. Judges have a precision/recall trade-off just like retrievers do.

## 7. Metric 2 — Answer Relevancy (does it address the question?)

Faithfulness can be *fooled*: an answer that just parrots a retrieved chunk is perfectly grounded but may **ignore the question**. Answer Relevancy catches that.

**Algorithm (RAGAS-style reverse-question trick):** from the *answer*, generate the questions it would be a good answer *to*, then measure how close those are to the **original** question. An on-topic answer regenerates questions close to the original; an evasive or off-topic answer doesn't.

Our keyless `gen_questions` is a template proxy; the real hook asks an LLM to produce N candidate questions. We score with cosine over bag-of-words (swap in a dense embedder from L85 in production).

> **Honesty note:** BoW cosine is vocabulary-sensitive (the exact problem L88 attacked). We keep the golden answers sharing vocabulary with their questions so the demo is clean — but *this is precisely why real relevancy metrics use semantic embeddings, not word overlap.*

In [ ]:
def gen_questions(answer_text: str, n: int = 3) -> List[str]:
    # Proxy for 'LLM, write questions this answer responds to'.
    # Deterministic: build questions from the answer's content words.
    kw = tokens(answer_text)
    if not kw:
        return ["?"]
    # take salient chunks of keywords to synthesize pseudo-questions
    qs = []
    for i in range(n):
        window = kw[i:i+5] if len(kw) > i else kw[:5]
        qs.append("what about " + " ".join(window))
    return qs

def answer_relevancy(question: str, answer_text: str, n: int = 3) -> float:
    if "don't know" in answer_text.lower():
        return 0.0   # a refusal is not 'relevant' to a question it declined to answer
    gq = gen_questions(answer_text, n)
    sims = [cosine_bow(question, q) for q in gq]
    return sum(sims) / len(sims) if sims else 0.0

on_topic  = run_system(GOLDEN[0])["answer"]                 # about sour/under-extraction
off_topic = "The machine should warm up for twenty minutes before the group head is hot."
print("relevancy, on-topic answer to 'why is my espresso sour?' :",
      round(answer_relevancy("why is my espresso sour?", on_topic), 3))
print("relevancy, off-topic answer to same question             :",
      round(answer_relevancy("why is my espresso sour?", off_topic), 3))

# 💡 EXPERIMENT: write an answer that repeats the question's words but says nothing useful.
# You'll see relevancy stays high — proof that NO single metric is sufficient. You need the quad.

## 8. Metrics 3 & 4 — Context Precision & Context Recall

Faithfulness and Relevancy grade the **generator**. These two grade the **retriever** — the other half of a RAG system. They need the golden `ground_context` labels.

**Context Recall** — *did we retrieve everything the ground-truth answer needs?*
Decompose the **ground-truth answer** into claims; a claim "counts" if some *retrieved* chunk supports it. `recall = supported ground-truth claims / total`. Low recall ⇒ the evidence simply wasn't fetched, so even a perfect generator can't answer.

**Context Precision@k** — *is the relevant stuff ranked high?* Retrieving the right doc at rank 5 behind four junk docs wastes context budget and invites distraction. RAGAS weights precision by rank:

$$\text{ContextPrecision} = \frac{\sum_{k} \big(\text{Precision@}k \cdot \mathbb{1}[\text{chunk }k\text{ relevant}]\big)}{\text{total relevant chunks}}$$

A chunk is "relevant" if it's in the golden `ground_context` (or, when labels are coarse, if it supports the ground-truth answer). Ranking the relevant chunk first scores **higher** than ranking it last — the metric rewards good ordering, which is exactly what your reranker from L86 optimizes.

In [ ]:
def context_recall(ground_truth: str, retrieved: List[str]) -> float:
    if "don't know" in ground_truth.lower():
        return 1.0  # nothing needed to be recalled for a refusal row
    claims = split_claims(ground_truth)
    if not claims:
        return 1.0
    return sum(claim_supported(c, retrieved) for c in claims) / len(claims)

def _is_relevant(doc_id: str, gold_ctx: List[str], ground_truth: str) -> bool:
    if gold_ctx:                       # trust explicit labels when we have them
        return doc_id in gold_ctx
    return overlap_coeff(ground_truth, CORPUS[doc_id]) >= 0.5

def context_precision(retrieved: List[str], gold_ctx: List[str], ground_truth: str) -> float:
    rel_flags = [_is_relevant(d, gold_ctx, ground_truth) for d in retrieved]
    total_rel = sum(rel_flags)
    if total_rel == 0:
        return 0.0
    running, hits, acc = 0, 0, 0.0
    for k, is_rel in enumerate(rel_flags, start=1):
        if is_rel:
            hits += 1
            acc += hits / k            # precision@k at this relevant position
    return acc / total_rel

# Show precision REWARDS ranking: same docs, relevant one first vs last.
gold = ["d_sour"]; gt = GOLDEN[0]["ground_truth"]
print("relevant doc at rank 1:", round(context_precision(["d_sour", "d_bitter"], gold, gt), 3))
print("relevant doc at rank 2:", round(context_precision(["d_bitter", "d_sour"], gold, gt), 3))

out = run_system(GOLDEN[3])            # g4 temperature question
print("g4 context recall     :", round(context_recall(GOLDEN[3]["ground_truth"], out["retrieved_context"]), 2))

## 9. LLM-as-Judge — scoring free text, and why judges are biased

The metrics above are *reference-based*: they compare against a golden answer with cheap lexical ops. But some qualities ("is this explanation *clear*? is the *tone* right?") have no lexical formula. For those you ask a **stronger LLM to score the answer against a rubric** — the *LLM-as-judge* pattern.

A pointwise judge prompt looks like this (this exact template is what the real hook would send):

```
You are grading an answer. Score 1-5 on FAITHFULNESS to the reference.
Question: {q}
Reference answer: {ground_truth}
Candidate answer: {candidate}
Rubric: 5=fully supported & complete; 3=partially; 1=contradicts/unsupported.
Return ONLY the integer.
```

**Judges are systematically biased — you must know these:**
- **Position bias** — in pairwise A-vs-B grading, models favor whichever answer is shown *first*. Mitigate by scoring both orderings and averaging.
- **Verbosity bias** — longer, more confident answers score higher even when wrong. Mitigate with rubrics that reward correctness, not length.
- **Self-preference bias** — a model rates *its own* style of output higher. Mitigate by using a *different* model family as judge than the one that generated.
- **Sycophancy** — the judge agrees with assertive phrasing. Mitigate by hiding which answer is the "candidate".

Our keyless judge is a deterministic proxy: it maps token-F1 against the reference onto a 1–5 rubric. The real hook (if a key is set) would call an actual model with the prompt above.

In [ ]:
def _token_f1(a: str, b: str) -> float:
    A, B = token_set(a), token_set(b)
    if not A or not B:
        return 0.0
    inter = len(A & B)
    if inter == 0:
        return 0.0
    p, r = inter / len(A), inter / len(B)
    return 2 * p * r / (p + r)

def llm_judge(question: str, ground_truth: str, candidate: str) -> int:
    # pointwise 1-5 faithfulness score; real hook -> actual model, else deterministic proxy
    if _real_llm_available():
        # Real path (only runs if a key exists). Left as an exercise / production hook.
        raise NotImplementedError("Wire your provider's chat call here using the rubric prompt above.")
    f1 = _token_f1(ground_truth, candidate)
    # map [0,1] F1 -> {1..5}
    return max(1, min(5, 1 + round(f1 * 4)))

for r in GOLDEN[:4]:
    out = run_system(r)
    print(f"[{r['id']}] judge score {llm_judge(r['question'], r['ground_truth'], out['answer'])}/5  "
          f"| answer: {out['answer'][:55]}...")

## 10. Calibrating the judge against humans (callback to L29)

A judge you haven't validated is just *another* unmeasured model. Before you trust its scores, you **calibrate**: collect human labels on a sample, then measure **correlation** between judge scores and human scores. If they don't correlate, your rubric is broken — fix the rubric, don't ship the judge.

We attach a small `human_label` (1–5) to each row — simulating a human rater — and compute **Spearman rank correlation** between judge and human. Rule of thumb: ρ > 0.7 means the judge tracks human judgment well enough to use as a proxy; below that, iterate on the rubric.

This is the same *calibration* mindset from L29 (a confident model must be a *correct* model): a metric is only trustworthy once its agreement with ground-truth reality is measured, not assumed.

In [ ]:
from scipy.stats import spearmanr

# Simulated human ratings (in real life: a person grades these rows once).
HUMAN_LABEL = {"g1": 5, "g2": 5, "g3": 1, "g4": 5, "g5": 1}

judge_scores, human_scores = [], []
for r in GOLDEN:
    out = run_system(r)
    judge_scores.append(llm_judge(r["question"], r["ground_truth"], out["answer"]))
    human_scores.append(HUMAN_LABEL[r["id"]])

rho, _ = spearmanr(judge_scores, human_scores)
print("judge scores:", judge_scores)
print("human scores:", human_scores)
print(f"Spearman correlation judge vs human: {rho:.2f}",
      "-> TRUST the judge" if rho > 0.7 else "-> rubric needs work")

# 💡 EXPERIMENT: change HUMAN_LABEL['g4'] to 1 (pretend a human disagreed).
# Watch the correlation drop — this is how you'd DETECT a judge whose rubric is off.

## 11. Putting it together — the evaluation scoreboard

A golden dataset is only useful when you can run **all metrics over all rows** and get one table you can diff between versions. This function is the seed of the reusable harness we'll finish in the L96 capstone.

Read the scoreboard as a **quadrant**:
- Low **context recall/precision** ⇒ fix the *retriever* (chunking L84, hybrid L85, rerank L86).
- Low **faithfulness** ⇒ fix *grounding* (L87) — the generator is inventing.
- Low **answer relevancy** ⇒ the generator is grounded but *dodging the question*.

Look for `g3` (hallucination) tanking **faithfulness** and `g5` (should-have-refused) tanking **faithfulness + relevancy** while its retrieval looks fine — the metrics *localize the bug* for you.

In [ ]:
def evaluate(golden) -> Dict:
    rows = []
    for r in golden:
        out = run_system(r)
        ctx, ans = out["retrieved_context"], out["answer"]
        rows.append({
            "id": r["id"],
            "faithfulness":   round(faithfulness(ans, ctx), 2),
            "answer_relev":   round(answer_relevancy(r["question"], ans), 2),
            "ctx_precision":  round(context_precision(ctx, r["ground_context"], r["ground_truth"]), 2),
            "ctx_recall":     round(context_recall(r["ground_truth"], ctx), 2),
        })
    agg = {m: round(sum(row[m] for row in rows) / len(rows), 3)
           for m in ["faithfulness", "answer_relev", "ctx_precision", "ctx_recall"]}
    return {"rows": rows, "aggregate": agg}

report = evaluate(GOLDEN)
hdr = f"{'id':<5}{'faith':>8}{'relev':>8}{'ctx_p':>8}{'ctx_r':>8}"
print(hdr); print("-" * len(hdr))
for row in report["rows"]:
    print(f"{row['id']:<5}{row['faithfulness']:>8}{row['answer_relev']:>8}"
          f"{row['ctx_precision']:>8}{row['ctx_recall']:>8}")
print("-" * len(hdr))
a = report["aggregate"]
print(f"{'AGG':<5}{a['faithfulness']:>8}{a['answer_relev']:>8}{a['ctx_precision']:>8}{a['ctx_recall']:>8}")

## 12. Verification checklist

Everything below is deterministic and runs keyless. If any line prints ❌ the lesson's logic is broken. This is the same discipline as your L90 CI gate — an eval you can't re-run isn't an eval.

In [ ]:
checks = []
def check(name, cond):
    checks.append(cond)
    print(("✅" if cond else "❌"), name)

# 1. grounded answer is fully faithful
g1 = run_system(GOLDEN[0])
check("faithfulness of grounded g1 == 1.0", faithfulness(g1["answer"], g1["retrieved_context"]) == 1.0)

# 2. hallucinated g3 has low faithfulness
g3 = run_system(GOLDEN[2])
check("faithfulness of hallucinated g3 < 0.5", faithfulness(g3["answer"], g3["retrieved_context"]) < 0.5)

# 3. context precision rewards ranking
gt = GOLDEN[0]["ground_truth"]
p_first = context_precision(["d_sour", "d_bitter"], ["d_sour"], gt)
p_last  = context_precision(["d_bitter", "d_sour"], ["d_sour"], gt)
check("context precision(rank1) > context precision(rank2)", p_first > p_last)

# 4. context recall high when the right doc is retrieved
g4 = run_system(GOLDEN[3])
check("context recall of g4 >= 0.5", context_recall(GOLDEN[3]["ground_truth"], g4["retrieved_context"]) >= 0.5)

# 5. answer relevancy discriminates on/off topic
on  = answer_relevancy("why is my espresso sour?", run_system(GOLDEN[0])["answer"])
off = answer_relevancy("why is my espresso sour?", "warm up the machine twenty minutes")
check("relevancy on-topic > off-topic", on > off)

# 6. judge correlates with humans
js = [llm_judge(r["question"], r["ground_truth"], run_system(r)["answer"]) for r in GOLDEN]
hs = [HUMAN_LABEL[r["id"]] for r in GOLDEN]
rho, _ = spearmanr(js, hs)
check("judge vs human Spearman > 0.7", rho > 0.7)

# 7. all scoreboard values are valid probabilities
allvals = [v for row in report["rows"] for k, v in row.items() if k != "id"]
check("all metric values in [0,1]", all(0.0 <= v <= 1.0 for v in allvals))

# 8. the golden set contains a refusal (out-of-domain) row
check("golden set has an out-of-domain refusal row", any(not r["ground_context"] for r in GOLDEN))

# 9. keyless: everything above ran with no API key
check("ran in keyless MOCK mode", not _real_llm_available())

print("\nRESULT:", "ALL PASS ✅" if all(checks) else "SOME FAILED ❌", f"({sum(checks)}/{len(checks)})")
assert all(checks), "Verification failed"


## 13. Recap & what's next

**What you built today** — the foundation of trustworthy AI systems:
- A **golden dataset**: a versioned, human-authored ruler of `(question, ground-truth, ground-context)` rows, deliberately including a refusal case.
- The **four RAG metrics from scratch** — Faithfulness & Answer Relevancy (grade the *generator*), Context Precision & Recall (grade the *retriever*). You now know what every number *means*, so when you later `pip install ragas` it's not a black box.
- **LLM-as-judge**: pointwise rubric scoring, the four systematic biases (position, verbosity, self-preference, sycophancy), and **calibration** against human labels via rank correlation — the L29 mindset applied to evaluation.
- A **scoreboard** that runs all metrics over the whole set and *localizes* failures to the right subsystem.

**The one idea to keep:** *you cannot improve what you don't measure, and you can't trust a measurement you haven't validated.* A metric is only as good as its correlation with reality.

**Next — Lesson 92: Regression & canary evals in CI.** Today's scoreboard is a snapshot. Next we make it a **gate that runs on every change**: baseline thresholds, catching *scoring drift* (when the judge itself silently changes), and canary rows that must never regress — turning this notebook into the CI check that protects your `rag-service`.

> 💡 **Stretch before next time:** add two new golden rows — one multi-hop bridge question (L89) and one vocabulary-mismatch question (L88) — and see which metric drops. That gap is your system's next improvement target.